In [ ]:
from datasets import load_dataset, Dataset
# import evaluate
import pandas as pd
from tqdm import tqdm
import os
os.environ["HF_TOKEN"] = "hf_token" # removed for safety reasons

from huggingface_hub import login
login(os.environ["HF_TOKEN"])


PRED_DATASET = "businessrules/Qwen_base_tuned_exp17_results"
output_path = "businessrules/Qwen_exp17_eval_meteor_bertscore"
# PRED_DATASET = "businessrules/GPT4_baseline_results"
# Results_dataset = "businessrules/GPT4_eval_meteor_bertscore"
eval_dataset = load_dataset(PRED_DATASET, split = "train")

references = [[x] for x in eval_dataset["golden_business_rule"]]
references_flat = eval_dataset["golden_business_rule"]

# base_predictions = eval_dataset["base_model_prediction"]
ft_predictions   = eval_dataset["finetuned_model_prediction"]


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


data.parquet: reconstructing file:   0%|          |  0.00B /  134kB            

data.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

In [ ]:
# =========================================================
# 0. Setup
# =========================================================
!pip install -q evaluate bert-score nltk datasets sentencepiece

import torch
import nltk
import evaluate
from datasets import load_dataset, Dataset

nltk.download("wordnet")
nltk.download("omw-1.4")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# =========================================================
# 1. Load evaluation dataset
# =========================================================
# PRED_DATASET = "businessrules/Qwen2.5-Coder-1.5B_base_tuned_exp1_results"
eval_dataset = load_dataset(PRED_DATASET, split="train")

# =========================================================
# 2. Load metrics
# =========================================================
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# =========================================================
# 3. Prepare references & predictions
# =========================================================
references = [[x] for x in eval_dataset["golden_business_rule"]]
references_flat = eval_dataset["golden_business_rule"]

# base_predictions = eval_dataset["base_model_prediction"]
ft_predictions   = eval_dataset["finetuned_model_prediction"]

# =========================================================
# 4. Compute per-example METEOR (manually, one by one)
# =========================================================
meteor_base = []
meteor_ft   = []


for pred, ref in zip(ft_predictions, references):
    score = meteor.compute(predictions=[pred], references=[ref])["meteor"]
    meteor_ft.append(score)

# =========================================================
# 5. BERTScore (BATCHED, GPU-SAFE)
# =========================================================
BERT_MODEL = "roberta-large"   
BATCH_SIZE = 8                

def batched_bertscore(preds, refs):
    all_p, all_r, all_f1 = [], [], []

    for i in range(0, len(preds), BATCH_SIZE):
        batch_preds = preds[i:i+BATCH_SIZE]
        batch_refs  = refs[i:i+BATCH_SIZE]

        scores = bertscore.compute(
            predictions=batch_preds,
            references=batch_refs,
            model_type=BERT_MODEL,
            lang="en",
            rescale_with_baseline=True,
            device=DEVICE
        )

        all_p.extend(scores["precision"])
        all_r.extend(scores["recall"])
        all_f1.extend(scores["f1"])

        # cleanup
        del scores
        torch.cuda.empty_cache()

    return all_p, all_r, all_f1


ft_p, ft_r, ft_f1 = batched_bertscore(
    ft_predictions,
    references_flat
)

# =========================================================
# 6. Build unified HF evaluation dataset
# =========================================================
rows = []

for i in range(len(references_flat)):
    rows.append({
        "id": i,
        "golden_business_rule": references_flat[i],

        # --------------------
        # Base model
        # --------------------
        # "base_prediction": base_predictions[i],
        # "base_meteor": meteor_base[i],
        # "base_bertscore_p": base_p[i],
        # "base_bertscore_r": base_r[i],
        # "base_bertscore_f1": base_f1[i],

        # --------------------
        # Fine-tuned model
        # --------------------
        "finetuned_prediction": ft_predictions[i],
        "finetuned_meteor": meteor_ft[i],
        "finetuned_bertscore_p": ft_p[i],
        "finetuned_bertscore_r": ft_r[i],
        "finetuned_bertscore_f1": ft_f1[i],
    })

eval_results_dataset = Dataset.from_list(rows)

# =========================================================
# 7. Push to Hugging Face Hub
# =========================================================
eval_results_dataset.push_to_hub(
    output_path,
    split="test"
)

print("METEOR + BERTScore evaluation completed and uploaded.")


In [ ]:
import numpy as np

# Base model
# base_meteor = np.array([x["base_meteor"] for x in eval_results_dataset])
# base_bertscore_f1 = np.array([x["base_bertscore_f1"] for x in eval_results_dataset])

# Fine-tuned model
ft_meteor = np.array([x["finetuned_meteor"] for x in eval_results_dataset])
ft_bertscore_f1 = np.array([x["finetuned_bertscore_f1"] for x in eval_results_dataset])

def summarize(metric_array, name):
    print(f"{name}: mean={metric_array.mean():.4f}, std={metric_array.std():.4f}, min={metric_array.min():.4f}, max={metric_array.max():.4f}")

# print("=== Base Model ===")
# summarize(base_meteor, "METEOR")
# summarize(base_bertscore_f1, "BERTScore F1")

print("=== Fine-Tuned Model ===")
summarize(ft_meteor, "METEOR")
summarize(ft_bertscore_f1, "BERTScore F1")


=== Fine-Tuned Model ===
METEOR: mean=0.3650, std=0.1591, min=0.0347, max=0.9101
BERTScore F1: mean=0.3043, std=0.2028, min=-0.3825, max=0.9043


In [ ]:
## BertScore and METEOR for GPT-4.1
from datasets import load_dataset, Dataset
# import evaluate
import pandas as pd
from tqdm import tqdm
import os

from huggingface_hub import login
login(os.environ["HF_TOKEN"])


# PRED_DATASET = "businessrules/Qwen_base_tuned_exp16_results"
# output_path = "businessrules/Qwen_exp16_eval_meteor_bertscore"
PRED_DATASET = "businessrules/GPT4_baseline_results"
Results_dataset = "businessrules/GPT4_eval_meteor_bertscore"
eval_dataset = load_dataset(PRED_DATASET, split = "train")
# =========================================================
# 0. Setup
# =========================================================
!pip install -q evaluate bert-score nltk datasets sentencepiece

import torch
import nltk
import evaluate
from datasets import load_dataset, Dataset

nltk.download("wordnet")
nltk.download("omw-1.4")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# =========================================================
# 1. Load evaluation dataset
# =========================================================
PRED_DATASET = "businessrules/GPT4_baseline_results"
output_path = "businessrules/GPT4_eval_meteor_bertscore"


# =========================================================
# 2. Load metrics
# =========================================================
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")

# =========================================================
# 3. Prepare references & predictions
# =========================================================
references = [[x] for x in eval_dataset["golden_business_rule"]]
references_flat = eval_dataset["golden_business_rule"]

gpt4_predictions = eval_dataset["gpt4_prediction"]

# =========================================================
# 4. Compute per-example METEOR (manually, one by one)
# =========================================================
meteor_gpt4 = []

for pred, ref in zip(gpt4_predictions, references):
    score = meteor.compute(predictions=[pred], references=[ref])["meteor"]
    meteor_gpt4.append(score)

# =========================================================
# 5. BERTScore (BATCHED, GPU-SAFE)
# =========================================================
BERT_MODEL = "roberta-large"  
BATCH_SIZE = 8                

def batched_bertscore(preds, refs):
    all_p, all_r, all_f1 = [], [], []

    for i in range(0, len(preds), BATCH_SIZE):
        batch_preds = preds[i:i+BATCH_SIZE]
        batch_refs  = refs[i:i+BATCH_SIZE]

        scores = bertscore.compute(
            predictions=batch_preds,
            references=batch_refs,
            model_type=BERT_MODEL,
            lang="en",
            rescale_with_baseline=True,
            device=DEVICE
        )

        all_p.extend(scores["precision"])
        all_r.extend(scores["recall"])
        all_f1.extend(scores["f1"])

        # cleanup
        del scores
        torch.cuda.empty_cache()

    return all_p, all_r, all_f1

gpt4_p, gpt4_r, gpt4_f1 = batched_bertscore(
    gpt4_predictions,
    references_flat
)

# =========================================================
# 6. Build unified HF evaluation dataset
# =========================================================
rows = []

for i in range(len(references_flat)):
    rows.append({
        "id": i,
        "golden_business_rule": references_flat[i],

        # --------------------
        # GPT-4
        # --------------------
        "gpt4_prediction": gpt4_predictions[i],
        "gpt4_meteor": meteor_gpt4[i],
        "gpt4_bertscore_p": gpt4_p[i],
        "gpt4_bertscore_r": gpt4_r[i],
        "gpt4_bertscore_f1": gpt4_f1[i],
    })

eval_results_dataset = Dataset.from_list(rows)

# =========================================================
# 7. Push to Hugging Face Hub
# =========================================================
eval_results_dataset.push_to_hub(
    output_path,
    split="test"
)

print(" METEOR + BERTScore evaluation completed and uploaded.")

In [ ]:
import numpy as np

# GPT-4.1
gpt4_meteor = np.array([x["gpt4_meteor"] for x in eval_results_dataset])
gpt4_bertscore_f1 = np.array([x["gpt4_bertscore_f1"] for x in eval_results_dataset])

def summarize(metric_array, name):
    print(f"{name}: mean={metric_array.mean():.4f}, std={metric_array.std():.4f}, min={metric_array.min():.4f}, max={metric_array.max():.4f}")

print("=== GPT-4 ===")
summarize(gpt4_meteor, "METEOR")
summarize(gpt4_bertscore_f1, "BERTScore F1")

=== GPT-4 ===
METEOR: mean=0.3460, std=0.1182, min=0.0472, max=0.7363
BERTScore F1: mean=0.1951, std=0.1732, min=-0.4548, max=0.6204
